In [1]:
# 

In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=mgfrlifLPTjqqz853VJJHE0PzMRcQQ&access_type=offline&code_challenge=XT8HEnInaBXQ5GUTCMzrxoHqZowW1OAadgvd5AzPYS0&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning

In [2]:
import numpy as np
from typing import Any
import os
import hail as hl
import pyspark.sql.functions as f
import pandas as pd

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.l2g_prediction import L2GPrediction
hail_dir = os.path.dirname(hl.__file__)
session = Session(hail_home=hail_dir, start_hail=True, extended_spark_conf={"spark.driver.memory": "12g","spark.kryoserializer.buffer.max": "500m","spark.driver.maxResultSize":"2g"})
hl.init(sc=session.spark.sparkContext, log="/dev/null")

Loading BokehJS ...

24/04/16 14:28:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://192.168.0.232:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null


In [9]:
path="/Users/yt4/Projects/ot_data/evidence"
df=session.spark.read.parquet(path)
df.printSchema()

root
 |-- datasourceId: string (nullable = true)
 |-- targetId: string (nullable = true)
 |-- alleleOrigins: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- allelicRequirements: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ancestry: string (nullable = true)
 |-- ancestryId: string (nullable = true)
 |-- beta: double (nullable = true)
 |-- betaConfidenceIntervalLower: double (nullable = true)
 |-- betaConfidenceIntervalUpper: double (nullable = true)
 |-- biologicalModelAllelicComposition: string (nullable = true)
 |-- biologicalModelGeneticBackground: string (nullable = true)
 |-- biologicalModelId: string (nullable = true)
 |-- biomarkerName: string (nullable = true)
 |-- biomarkers: struct (nullable = true)
 |    |-- geneExpression: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |-- variant

In [5]:
df.count()

17317290

In [4]:
EFOs=[
    "EFO_0003767", #https://www.ebi.ac.uk/gwas/efotraits/EFO_0003767
    "EFO_0000384", #https://www.ebi.ac.uk/gwas/efotraits/EFO_0000384
    "EFO_0000729", #https://www.ebi.ac.uk/gwas/efotraits/EFO_0000729
    "EFO_0000555", #https://www.ebi.ac.uk/gwas/efotraits/EFO_0000555 
]

In [14]:

df_filtered = df.filter(df.diseaseFromSourceMappedId.isin(EFOs))
df_filtered=df_filtered.filter(df_filtered.clinicalPhase.isNotNull())
df_filtered.count()

2724

In [15]:
df_filtered.show(3)

+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+------------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+------+---------+----------------------+---------------+----------+------------+-----------+-------------+----+------------------------+--------

In [16]:
category_counts = df_filtered.groupBy("clinicalPhase").count().show()
print(category_counts)

+-------------+-----+
|clinicalPhase|count|
+-------------+-----+
|          1.0|  250|
|          4.0|  973|
|          0.5|   34|
|          3.0|  641|
|          2.0|  826|
+-------------+-----+

None


In [17]:
category_counts = df_filtered.groupBy("clinicalSignificances").count().show()
print(category_counts)

+---------------------+-----+
|clinicalSignificances|count|
+---------------------+-----+
|                 null| 2724|
+---------------------+-----+

None


In [18]:
category_counts = df_filtered.groupBy("clinicalStatus").count().show()
print(category_counts)

+--------------------+-----+
|      clinicalStatus|count|
+--------------------+-----+
|           Suspended|   33|
|      Unknown status|  112|
|           Completed| 1127|
|Enrolling by invi...|    8|
|  Not yet recruiting|  103|
|          Recruiting|  357|
|          Terminated|  202|
|           Withdrawn|  111|
|Active, not recru...|   74|
|                null|  597|
+--------------------+-----+

None


In [20]:
category_counts = df_filtered.groupBy("datasourceId").count().show()
print(category_counts)

+------------+-----+
|datasourceId|count|
+------------+-----+
|      chembl| 2724|
+------------+-----+

None


In [21]:
df_filtered.show(2)

+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+------------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+------+---------+----------------------+---------------+----------+------------+-----------+-------------+----+------------------------+--------

In [22]:
df_tmp = df_filtered.dropDuplicates(["targetId"])
df_tmp.count()

296

In [23]:
df_aggregated = df_filtered.groupBy("targetId").agg({"clinicalPhase": "max"})

In [24]:
df_aggregated.show(2)

+---------------+------------------+
|       targetId|max(clinicalPhase)|
+---------------+------------------+
|ENSG00000198851|               3.0|
|ENSG00000113851|               3.0|
+---------------+------------------+
only showing top 2 rows



In [25]:
category_counts = df_aggregated.groupBy("max(clinicalPhase)").count().show()
print(category_counts)

+------------------+-----+
|max(clinicalPhase)|count|
+------------------+-----+
|               1.0|   24|
|               4.0|   88|
|               3.0|   25|
|               2.0|  159|
+------------------+-----+

None


In [26]:
df_aggregated.toPandas().to_csv("/Users/yt4/Projects/ot_data/24_03_ppp/IBD_max_clin_phase.tsv", sep="\t", index=False)

# CAD

In [32]:

EFOs=[
"EFO_0000319",
"EFO_0000275",
"HP_0002140",
"EFO_0000537",
"EFO_0004269"
"EFO_0000612",
"MONDO_0019171"
"EFO_0003777"
]

In [33]:
df_filtered = df.filter(df.diseaseFromSourceMappedId.isin(EFOs))
df_filtered=df_filtered.filter(df_filtered.clinicalPhase.isNotNull())
df_filtered.count()
df_aggregated = df_filtered.groupBy("targetId").agg({"clinicalPhase": "max"})
category_counts = df_aggregated.groupBy("max(clinicalPhase)").count().show()
print(category_counts)

+------------------+-----+
|max(clinicalPhase)|count|
+------------------+-----+
|               1.0|   14|
|               4.0|  256|
|               0.5|    3|
|               3.0|   94|
|               2.0|   29|
+------------------+-----+

None


In [34]:
df_aggregated.toPandas().to_csv("/Users/yt4/Projects/ot_data/24_03_ppp/CAD_max_clin_phase.tsv", sep="\t", index=False)